In [3]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "WHQ_D.xpt"
csv_file = DATA_DIR / "WHQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 WHQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WHD010",
    "WHD020",
    "WHQ030",
    "WHQ040",
    "WHD050",
    "WHQ060",
    "WHQ070",
    "WHD080A",
    "WHD080B",
    "WHD080C",
    "WHD080D",
    "WHD080E",
    "WHD080F",
    "WHD080G",
    "WHD080H",
    "WHD080I",
    "WHD080J",
    "WHD080K",
    "WHD080L",
    "WHD080M",
    "WHD080N",
    "WHD080O",
    "WHD080P",
    "WHD080Q",
    "WHD080R",
    "WHD080S",
    "WHQ270",
    "WHQ280A",
    "WHQ280B",
    "WHQ280C",
    "WHQ280D",
    "WHQ280E",
    "WHQ090",
    "WHD100A",
    "WHD100B",
    "WHD100C",
    "WHD100D",
    "WHD100E",
    "WHD100F",
    "WHD100G",
    "WHD100H",
    "WHD100I",
    "WHD100J",
    "WHD100K",
    "WHD100L",
    "WHD100M",
    "WHD100N",
    "WHD100O",
    "WHD100P",
    "WHD100Q",
    "WHD100R",
    "WHD100S",
    "WHQ210",
    "WHD220",
    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140",
    "WHQ150"
]


if len(df) != 6139:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 60:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "WHQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: WHQ_D structure = "
    "6,139 rows x 60 columns."
)


# ============================================================
# 4. FIELD TYPES
# ============================================================

#
# These are released height/weight measurement fields.
#
# Even though the current XPT values are mathematically
# whole numbers, preserve them as floats:
#
# 180.0 -> 180.0
# 68.0  -> 68.0
#
# Do NOT Int64 these fields.
#

MEASUREMENT_COLUMNS = [
    "WHD010",
    "WHD020",
    "WHD050",
    "WHD220",
    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140"
]


INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in MEASUREMENT_COLUMNS
]


print("\nMeasurement fields preserved as float:")
print(MEASUREMENT_COLUMNS)

print(
    "Integer/code/count/age fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXPECTED TINY VALUES
# ============================================================

EXPECTED_TINY_COUNTS = {
    "WHD220": 107
}


EXPECTED_TINY_TOTAL = 107


actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


if actual_tiny_columns != {"WHD220"}:

    raise ValueError(
        "STOP: Tiny XPORT values found in "
        "unexpected WHQ_D variable(s):\n"
        + str(
            sorted(
                actual_tiny_columns
            )
        )
    )


if int(
    tiny_counts["WHD220"]
) != 107:

    raise ValueError(
        "WHD220 tiny-value count differs "
        "from expected 107."
    )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: Exactly 107 tiny values "
    "found in WHD220 only."
)


# ============================================================
# 7. RESTORE VALIDATED WHD220 ZEROS
# ============================================================

mask = (
    df["WHD220"]
    .eq(TINY_VALUE)
)


print(
    "\nWHD220 tiny -> 0.0:",
    int(mask.sum())
)


df.loc[
    mask,
    "WHD220"
] = 0.0


if int(
    df["WHD220"]
    .eq(0)
    .sum()
) != 107:

    raise ValueError(
        "WHD220 zero restoration failed."
    )


print(
    "PASS: 107 WHD220 zeros restored."
)


# ============================================================
# 8. VERIFY NO TINY VALUE REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifact remains."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 9. GENUINE FRACTIONAL VALUE CHECK
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        fractional_columns[col] = count


if fractional_columns:

    raise ValueError(
        "Unexpected genuine fractional values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine fractional values = 0."
)

print(
    "NOTE: Measurement fields still remain float "
    "for released measurement fidelity."
)


# ============================================================
# 10. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn = df["SEQN"]


if not np.all(
    np.isclose(
        seqn.dropna(),
        np.round(
            seqn.dropna()
        ),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    seqn.isna().sum()
)


seqn_duplicates = int(
    seqn.duplicated().sum()
)


seqn_min = seqn.min()
seqn_max = seqn.max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:

    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:

    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31130:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41474:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. CDC/XPT MEASUREMENT RANGE CHECKS
# ============================================================

print(
    "\n--- MEASUREMENT RANGE CHECKS ---"
)


# WHD010
s = df["WHD010"]

if not (
    int(
        s.between(
            48,
            83
        ).sum()
    ) == 5963

    and

    int(
        s.eq(7777)
        .sum()
    ) == 2

    and

    int(
        s.eq(9999)
        .sum()
    ) == 108

    and

    int(
        s.isna()
        .sum()
    ) == 66
):

    raise ValueError(
        "WHD010 differs from expected source."
    )


print(
    "PASS: WHD010"
)


# WHD020
s = df["WHD020"]

if not (
    int(
        s.between(
            70,
            600
        ).sum()
    ) == 5994

    and

    int(
        s.eq(7777)
        .sum()
    ) == 4

    and

    int(
        s.eq(9999)
        .sum()
    ) == 91

    and

    int(
        s.isna()
        .sum()
    ) == 50
):

    raise ValueError(
        "WHD020 differs from expected source."
    )


print(
    "PASS: WHD020"
)


# WHD050
s = df["WHD050"]

if not (
    int(
        s.between(
            73,
            550
        ).sum()
    ) == 5987

    and

    int(
        s.eq(7777)
        .sum()
    ) == 6

    and

    int(
        s.eq(9999)
        .sum()
    ) == 96

    and

    int(
        s.isna()
        .sum()
    ) == 50
):

    raise ValueError(
        "WHD050 differs from expected source."
    )


print(
    "PASS: WHD050"
)


# WHD220
s = df["WHD220"]

if not (
    int(
        s.between(
            0,
            220
        ).sum()
    ) == 3475

    and

    int(
        s.eq(0)
        .sum()
    ) == 107

    and

    int(
        s.eq(99999)
        .sum()
    ) == 27

    and

    int(
        s.isna()
        .sum()
    ) == 2637
):

    raise ValueError(
        "WHD220 differs from expected source."
    )


print(
    "PASS: WHD220"
)


# WHD110
s = df["WHD110"]

if not (
    int(
        s.between(
            70,
            400
        ).sum()
    ) == 3270

    and

    int(
        s.eq(7777)
        .sum()
    ) == 2

    and

    int(
        s.eq(9999)
        .sum()
    ) == 68

    and

    int(
        s.isna()
        .sum()
    ) == 2799
):

    raise ValueError(
        "WHD110 differs from expected source."
    )


print(
    "PASS: WHD110"
)


# WHD120
s = df["WHD120"]

if not (
    int(
        s.between(
            70,
            390
        ).sum()
    ) == 4049

    and

    int(
        s.eq(7777)
        .sum()
    ) == 2

    and

    int(
        s.eq(9999)
        .sum()
    ) == 116

    and

    int(
        s.isna()
        .sum()
    ) == 1972
):

    raise ValueError(
        "WHD120 differs from expected source."
    )


print(
    "PASS: WHD120"
)


# WHD130
s = df["WHD130"]

if not (
    int(
        s.between(
            51,
            80
        ).sum()
    ) == 2125

    and

    int(
        s.eq(7777)
        .sum()
    ) == 1

    and

    int(
        s.eq(9999)
        .sum()
    ) == 64

    and

    int(
        s.isna()
        .sum()
    ) == 3949
):

    raise ValueError(
        "WHD130 differs from expected source."
    )


print(
    "PASS: WHD130"
)


# WHD140
s = df["WHD140"]

if not (
    int(
        s.between(
            73,
            610
        ).sum()
    ) == 5341

    and

    int(
        s.eq(7777)
        .sum()
    ) == 6

    and

    int(
        s.eq(9999)
        .sum()
    ) == 202

    and

    int(
        s.isna()
        .sum()
    ) == 590
):

    raise ValueError(
        "WHD140 differs from expected source."
    )


print(
    "PASS: WHD140"
)


# ============================================================
# 12. KEY QUESTIONNAIRE FREQUENCIES
# ============================================================

#
# NOTE:
#
# CDC HTML currently shows WHQ060:
#
# 1 = 629
# 2 = 441
# Missing = 5069
#
# The released WHQ_D.xpt attached here actually contains:
#
# 1 = 629
# 2 = 442
# Missing = 5068
#
# For XPT -> CSV fidelity we preserve the released XPT.
#

EXPECTED_FREQUENCIES = {

    "WHQ030": {
        "counts": {
            1: 2977,
            2: 402,
            3: 2741,
            7: 3,
            9: 16
        },
        "missing": 0
    },

    "WHQ040": {
        "counts": {
            1: 624,
            2: 3437,
            3: 2068,
            7: 2,
            9: 8
        },
        "missing": 0
    },

    # --------------------------------------------------------
    # IMPORTANT CORRECTION
    # Use released XPT counts.
    # --------------------------------------------------------

    "WHQ060": {
        "counts": {
            1: 629,
            2: 442
        },
        "missing": 5068
    },

    "WHQ070": {
        "counts": {
            1: 1885,
            2: 3622,
            7: 2,
            9: 1
        },
        "missing": 629
    },

    "WHQ090": {
        "counts": {
            1: 599,
            2: 3024,
            7: 2
        },
        "missing": 2514
    },

    "WHQ210": {
        "counts": {
            1: 1054,
            2: 2567,
            7: 2,
            9: 2
        },
        "missing": 2514
    }
}


print(
    "\n--- QUESTIONNAIRE FREQUENCY CHECKS ---"
)


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: source frequency mismatch.\n"
            f"Expected: {spec['counts']}\n"
            f"Actual:   {actual_counts}"
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: missing count mismatch.\n"
            f"Expected: {spec['missing']:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col:<8} "
        f"valid={sum(actual_counts.values()):,} "
        f"missing={actual_missing:,}"
    )


print(
    "PASS: Questionnaire frequencies "
    "match released XPT."
)


# ============================================================
# 13. CHECK WHQ150
# ============================================================

s = df["WHQ150"]


if not (
    int(
        s.between(
            10,
            85
        ).sum()
    ) == 5431

    and

    int(
        s.eq(99999)
        .sum()
    ) == 26

    and

    int(
        s.isna()
        .sum()
    ) == 682
):

    raise ValueError(
        "WHQ150 differs from expected source."
    )


print(
    "PASS: WHQ150 matches source."
)


# ============================================================
# 14. BUILD EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# Expected output =
#
# original released XPT
#
# +
# ONLY:
#
# WHD220 tiny -> 0.0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WHD220"]
    .eq(TINY_VALUE),
    "WHD220"
] = 0.0


# ============================================================
# 15. CLEAN ONLY INTEGER/CODE FIELDS
# ============================================================

#
# DO NOT Int64 the height/weight measurement fields.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Identifier/code/count/age fields "
    "converted to clean integer formatting."
)

print(
    "PASS: Height/weight measurement fields "
    "left as source-decoded floats."
)


# ============================================================
# 16. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

print(
    "\n--- PRE-EXPORT NUMERIC FIDELITY ---"
)


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in MEASUREMENT_COLUMNS:

        current_values = (
            df[col]
            .to_numpy()
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Values match expected XPT after "
    "ONLY WHD220 zero restoration."
)


# ============================================================
# 17. EXPORT CSV
# ============================================================

#
# NO:
#
# float_format
# round()
# recalculation
#
# Measurement fields retain source float representation.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 18. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 19. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 20. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nTotal numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All CSV numeric values "
    "match expected XPT values exactly."
)


# ============================================================
# 21. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 22. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(
        reader
    )


if len(raw_rows) != 6139:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 23. STRICT RAW CSV FIDELITY
# ============================================================

#
# Integer/code fields:
#
# 1.0 -> "1"
#
# Measurement fields:
#
# 180.0 -> "180.0"
# 68.0  -> "68.0"
# 0.0   -> "0.0"
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in MEASUREMENT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "WHQ_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV representation "
    "matches expected source formatting."
)


# ============================================================
# 24. UNWANTED .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

unwanted_dot_zero_count = 0
unwanted_dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            unwanted_dot_zero_count += 1


            if len(
                unwanted_dot_zero_examples
            ) < 20:

                unwanted_dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nUnwanted integer-field '.0' values:",
    unwanted_dot_zero_count
)


if unwanted_dot_zero_count != 0:

    print(
        "Examples:",
        unwanted_dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 remains in integer fields."
    )


print(
    "PASS: No unwanted .0 in integer/code fields."
)


# ============================================================
# 25. MEASUREMENT .0 PRESERVATION CHECK
# ============================================================

measurement_dot_zero_counts = {}


for col in MEASUREMENT_COLUMNS:

    count = sum(
        row[col].endswith(".0")
        for row in raw_rows
        if row[col] != ""
    )


    measurement_dot_zero_counts[col] = count


print(
    "\n--- LEGITIMATE MEASUREMENT .0 VALUES ---"
)


for col, count in measurement_dot_zero_counts.items():

    print(
        f"{col:<8}: {count:,}"
    )


expected_measurement_nonmissing = {
    col: int(
        expected_df[col]
        .notna()
        .sum()
    )
    for col in MEASUREMENT_COLUMNS
}


for col in MEASUREMENT_COLUMNS:

    if (
        measurement_dot_zero_counts[col]
        !=
        expected_measurement_nonmissing[col]
    ):

        raise ValueError(
            f"{col}: source-like .0 formatting changed."
        )


print(
    "PASS: Measurement .0 formatting preserved."
)


# ============================================================
# 26. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 27. FINAL TINY ARTIFACT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 28. FINAL WHD220 ZERO CHECK
# ============================================================

final_whd220_zeros = int(
    csv_df["WHD220"]
    .eq(0)
    .sum()
)


print(
    "\nWHD220 zeros in final CSV:",
    final_whd220_zeros
)


if final_whd220_zeros != 107:

    raise ValueError(
        "Final WHD220 zero count differs."
    )


print(
    "PASS: Final WHD220 zeros = 107."
)


# ============================================================
# 29. FINAL WHQ060 CHECK
# ============================================================

final_whq060 = (
    csv_df["WHQ060"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


final_whq060 = {
    int(k): int(v)
    for k, v
    in final_whq060.items()
}


if final_whq060 != {
    1: 629,
    2: 442
}:

    raise ValueError(
        "WHQ060 changed after CSV export."
    )


if int(
    csv_df["WHQ060"]
    .isna()
    .sum()
) != 5068:

    raise ValueError(
        "WHQ060 missing count changed."
    )


print(
    "PASS: Final WHQ060 preserves released XPT:"
)

print(
    "      1 = 629"
)

print(
    "      2 = 442"
)

print(
    "      Missing = 5,068"
)


# ============================================================
# 30. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("WHQ_D CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "WHD220:",
    107
)

print(
    "TOTAL:",
    107
)


print(
    "\nMeasurement fields preserved as float:"
)

for col in MEASUREMENT_COLUMNS:

    print(
        col
    )


print(
    "\nGenuine fractional values:",
    0
)


print(
    "\nWHQ060 released XPT frequencies:"
)

print(
    "1 = 629"
)

print(
    "2 = 442"
)

print(
    "Missing = 5,068"
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted integer .0 values:",
    unwanted_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "WHQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 WHQ_D XPT -> CSV
Rows: 6,139
Columns: 60
PASS: WHQ_D structure = 6,139 rows x 60 columns.

Measurement fields preserved as float:
['WHD010', 'WHD020', 'WHD050', 'WHD220', 'WHD110', 'WHD120', 'WHD130', 'WHD140']
Integer/code/count/age fields: 52

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 107

Tiny values by variable:
WHD220    107

PASS: Exactly 107 tiny values found in WHD220 only.

WHD220 tiny -> 0.0: 107
PASS: 107 WHD220 zeros restored.

Remaining tiny artifacts: 0
PASS: No tiny artifacts remain.

--- FRACTIONAL VALUE CHECK ---
PASS: Genuine fractional values = 0.
NOTE: Measurement fields still remain float for released measurement fidelity.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 31130.0
Maximum: 41474.0
PASS: SEQN validated.

--- MEASUREMENT RANGE CHECKS ---
PASS: WHD010
PASS: WHD020
PASS: WHD050
PASS: WHD220
PASS: WHD110
PASS: WHD120
PASS: WHD130
PASS: WHD140

--- QUESTIONNAIRE FREQUENCY CHECKS ---
PASS: WHQ030 